# A2 · OneHotEncoder from scratch: unknown categories and `drop`

**Type**: Implement from scratch · **Level**: ●●○ independent · **Time**: about 35 minutes · **core path ★**

## Objective

Implement a one-hot encoder that learns the categories on the training set, always produces the same columns, handles
never-seen categories and supports `drop="first"`, checking it against `sklearn.preprocessing.OneHotEncoder`.

## Dataset

`ds.loans(with_missing=False)`: two nominal columns, `city` (12 levels) and `contract_type` (4), as an array of
strings. In the test set the first five rows have the city replaced by `"city_XX"`, **never seen** in training.

## Educational purpose

See what `fit` hides: the sorted list of the categories of the **training set** decides the columns forever. And understand why
an unknown category and `drop` do not combine: the row of zeros would be ambiguous with the dropped category.

## Task

1. Write `MyOneHot(drop=None)`. `fit(X)`: for each column save the sorted categories (`np.unique`) in `categories_`.
2. `transform(X)` returns a `float` array with one column for every category of every feature, in the order of the features. An unknown category produces zeros in all the columns of that feature, **except** if `drop="first"`: in that case it raises `ValueError`.
3. With `drop="first"` drop the first column of each feature.
4. Write `names(feature_names)`: the list of the column names, `featurename_category`.
5. Check the result against `OneHotEncoder(sparse=False, handle_unknown="ignore")` (on the test with the unknown category) and against `OneHotEncoder(sparse=False, drop="first")` (on the clean test, without unknowns).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

df = ds.loans(with_missing=False)
cat = df[["city", "contract_type"]].to_numpy(dtype=object)
cat_tr, cat_te = train_test_split(cat, test_size=0.25, random_state=0)
cat_te_clean = cat_te.copy()
cat_te = cat_te.copy()
cat_te[:5, 0] = "city_XX"                          # a city never seen in training

## Your work

In [ ]:
class MyOneHot:
    def __init__(self, drop=None):
        self.drop = drop

    def fit(self, X):
        ...
        return self

    def transform(self, X):
        ...

    def names(self, feature_names):
        ...

In [ ]:
mine = MyOneHot().fit(cat_tr)
ref = OneHotEncoder(sparse=False, handle_unknown="ignore").fit(cat_tr)
mine_d = MyOneHot(drop="first").fit(cat_tr)
ref_d = OneHotEncoder(sparse=False, drop="first").fit(cat_tr)
mine.transform(cat_te).shape

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For each feature j: `out = (X[:, j].reshape(-1, 1) == self.categories_[j].reshape(1, -1)).astype(float)` (comparison by broadcasting); if `drop == "first"` discard `out[:, 1:]`. Concatenate with `np.hstack`. To recognize the unknowns: `~np.isin(X[:, j], self.categories_[j])`.'):
    check(close(mine.transform(cat_tr), ref.transform(cat_tr)), "matches scikit-learn on the training set",
          "the columns must follow the order (feature, then sorted categories)")
    check(close(mine.transform(cat_te), ref.transform(cat_te)), "matches scikit-learn on the test set with the unknown city",
          "a never-seen category must give zeros in all the columns of that feature")
    check(mine.transform(cat_te)[:5, :12].sum() == 0, "the rows with the unknown city have 12 zeros in the city columns",
          "check the first five rows of the test set")
    check(list(mine.names(["city", "contract_type"])) == list(ref.get_feature_names_out(["city", "contract_type"])),
          "the column names match", "name = feature + '_' + category")
    check(close(mine_d.transform(cat_tr), ref_d.transform(cat_tr)) and mine_d.transform(cat_tr).shape[1] == 14,
          "drop='first' removes one column per feature (16 - 2 = 14)", "with drop='first' 14 columns remain")
    check(close(mine_d.transform(cat_te_clean), ref_d.transform(cat_te_clean)), "drop='first' matches on the clean test set", "review drop")
    try:
        mine_d.transform(cat_te)
        raised = False
    except ValueError:
        raised = True
    check(raised, "drop='first' with an unknown category raises ValueError",
          "with drop='first' the row of zeros would be ambiguous: raise ValueError")
done()

---
The commented solution is in `solutions/A_from_scratch/A2_one_hot_encoder_from_scratch.ipynb`. Open it after you pass the check, or after you have really tried.